In [ ]:
!pip install -q faiss-cpu sentence-transformers spacy
!python -m spacy download en_core_web_trf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 31.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

# Load skill list from CSV
df = pd.read_csv("/content/drive/MyDrive/cleaned_skill_data.csv")
skills = df['skill'].astype(str).tolist()

# Load transformer model
model = SentenceTransformer("all-mpnet-base-v2")

# Generate embeddings
embeddings = model.encode(skills, show_progress_bar=True)

# Save embeddings
embeddings_df = pd.DataFrame(embeddings)
embeddings_df.to_csv("skill2vec.csv", index=False)

print("✅ Skill embeddings generated and saved to skill2vec.csv")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1536 [00:00<?, ?it/s]

✅ Skill embeddings generated and saved to skill2vec.csv


In [ ]:
import faiss
import numpy as np

# Load skill vectors
embedding_data = pd.read_csv("skill2vec.csv").values.astype('float32')

# Convert to C-contiguous format for FAISS
embedding_data = np.ascontiguousarray(embedding_data)

# Normalize and index
faiss.normalize_L2(embedding_data)
index = faiss.IndexFlatIP(embedding_data.shape[1])
index.add(embedding_data)

# Save index
faiss.write_index(index, "skill_index.faiss")
print("✅ FAISS index created and saved as skill_index.faiss")


✅ FAISS index created and saved as skill_index.faiss


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

# Load only 'skill' column from cleaned_skill_data.csv
df = pd.read_csv("/content/drive/MyDrive/cleaned_skill_data.csv")
skills = df['skill'].astype(str).tolist()

# Load sentence transformer
model = SentenceTransformer("all-mpnet-base-v2")

# Generate embeddings
embeddings = model.encode(skills, show_progress_bar=True)

# Save skill list for index search
pd.Series(skills).to_csv("skill_list.csv", index=False, header=False)

# Save only embeddings as clean float32 matrix
embeddings_df = pd.DataFrame(np.array(embeddings, dtype='float32'))
embeddings_df.to_csv("skill2vec.csv", index=False)

print("✅ Cleaned skill list + float32 embeddings saved.")


Batches:   0%|          | 0/1536 [00:00<?, ?it/s]

✅ Cleaned skill list + float32 embeddings saved.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

# Load clean data
skills = pd.read_csv("skill_list.csv", header=None)[0].tolist()
skill_vectors = pd.read_csv("skill2vec.csv").values.astype('float32')
skill_vectors = np.ascontiguousarray(skill_vectors)

# Load FAISS index
index = faiss.read_index("skill_index.faiss")

# Load SentenceTransformer
model = SentenceTransformer("all-mpnet-base-v2")

# Ask for a test query
query = input("✍️ Enter a job requirement or resume line: ")

# Embed and normalize
query_vec = model.encode([query]).astype('float32')
faiss.normalize_L2(query_vec)

# Search top 5 skills
D, I = index.search(query_vec, k=5)
top_skills = [skills[i] for i in I[0]]

# Display
print("\n🔍 Top matching skills:")
for skill in top_skills:
    print(f"✅ {skill}")


✍️ Enter a job requirement or resume line: experience with docker and kubernetes

🔍 Top matching skills:
✅ kubernetes
✅ docker container
✅ docker
✅ linux containers
✅ containerbased


In [ ]:
while True:
    query = input("\n✍️ Enter a job requirement or resume line (or 'exit' to stop): ")
    if query.lower() == "exit":
        break

    query_vec = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vec)
    D, I = index.search(query_vec, k=5)
    top_skills = [skills[i] for i in I[0]]

    print("\n🔍 Top matching skills:")
    for skill in top_skills:
        print(f"✅ {skill}")



✍️ Enter a job requirement or resume line (or 'exit' to stop): experience with React and Redux

🔍 Top matching skills:
✅ react css redux
✅ react js
✅ reactjs
✅ react js developer
✅ reactjs developer

✍️ Enter a job requirement or resume line (or 'exit' to stop): worked on Azure cloud infrastructure

🔍 Top matching skills:
✅ azure cloud
✅ microsoft azure
✅ microsoft_azure
✅ windows azure
✅ ms azure

✍️ Enter a job requirement or resume line (or 'exit' to stop): knowledge of RESTful APIs and CI/CD pipelines

🔍 Top matching skills:
✅ restful service development
✅ api development and integration
✅ ci pipelines
✅ restful apis
✅ restful applications

✍️ Enter a job requirement or resume line (or 'exit' to stop): knowledge of RESTful APIs and CI/CD pipelines"

🔍 Top matching skills:
✅ restful service development
✅ api development and integration
✅ ci pipelines
✅ restful apis
✅ api development

✍️ Enter a job requirement or resume line (or 'exit' to stop): built projects using Flask and SQL



RAG engine to your resume vs job description matching system

In [ ]:
# Load everything once
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

skills = pd.read_csv("skill_list.csv", header=None)[0].tolist()
skill_vectors = pd.read_csv("skill2vec.csv").values.astype('float32')
skill_vectors = np.ascontiguousarray(skill_vectors)
index = faiss.read_index("skill_index.faiss")
model = SentenceTransformer("all-mpnet-base-v2")

def extract_skills(text):
    """Extracts relevant skills from the given text using FAISS + embeddings."""
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 5]
    extracted = set()
    for line in sentences:
        vec = model.encode([line]).astype('float32')
        faiss.normalize_L2(vec)
        D, I = index.search(vec, k=3)
        for i in I[0]:
            extracted.add(skills[i])
    return list(extracted)


Matching Function

In [ ]:
def compare_resume_to_jd(resume_text, jd_text):
    resume_skills = extract_skills(resume_text)
    jd_skills = extract_skills(jd_text)

    matched = list(set(resume_skills) & set(jd_skills))
    missing = list(set(jd_skills) - set(resume_skills))

    skill_score = len(matched) / len(jd_skills) if jd_skills else 0

    return {
        "resume_skills": resume_skills,
        "jd_skills": jd_skills,
        "matched_skills": matched,
        "missing_skills": missing,
        "score": round(skill_score * 100, 2)
    }


testing


In [ ]:
resume_text = """
Built multiple projects using React, Node.js, and MongoDB.
Familiar with Docker and container-based deployment.
Experience with version control using Git and GitHub.
"""

jd_text = """
Looking for a full-stack developer with experience in React.js, Node.js, REST APIs, Docker, and CI/CD.
"""

result = compare_resume_to_jd(resume_text, jd_text)

print("\n🎯 Skill Match Score:", result['score'], "%")
print("\n✅ Matching Skills:", result['matched_skills'])
print("❌ Missing Skills:", result['missing_skills'])



🎯 Skill Match Score: 44.44 %

✅ Matching Skills: ['react js developer', 'docker container', 'docker', 'reactjs developer']
❌ Missing Skills: ['node js', 'api development and integration', 'node javascript', 'full stack web developer', 'node java script']


In [ ]:

!pip install -q sentence-transformers faiss-cpu spacy PyMuPDF python-docx
!python -m spacy download en_core_web_trf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 4.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import files

print("📄 Upload your resume file:")
resume_file = files.upload()

print("📄 Upload your JD file:")
jd_file = files.upload()


📄 Upload your resume file:


KeyboardInterrupt: 

In [ ]:
import fitz  # PyMuPDF
import docx

def extract_text(path):
    if path.endswith(".pdf"):
        doc = fitz.open(path)
        return "\n".join([page.get_text() for page in doc])
    elif path.endswith(".docx"):
        return "\n".join([para.text for para in docx.Document(path).paragraphs])
    elif path.endswith(".txt"):
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        return ""


In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load model and data
skills = pd.read_csv("skill_list.csv", header=None)[0].tolist()
skill_vectors = pd.read_csv("skill2vec.csv").values.astype('float32')
skill_vectors = np.ascontiguousarray(skill_vectors)
index = faiss.read_index("skill_index.faiss")
model = SentenceTransformer("all-mpnet-base-v2")

def extract_skills(text, top_k=3):
    """Extract relevant skills from input text."""
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 5]
    found = set()
    for line in sentences:
        vec = model.encode([line]).astype('float32')
        faiss.normalize_L2(vec)
        _, I = index.search(vec, top_k)
        for i in I[0]:
            found.add(skills[i])
    return list(found)


In [ ]:
def compare_resume_to_jd(resume_path, jd_path):
    resume_text = extract_text(resume_path)
    jd_text = extract_text(jd_path)

    resume_skills = extract_skills(resume_text)
    jd_skills = extract_skills(jd_text)

    matched = sorted(set(resume_skills) & set(jd_skills))
    missing = sorted(set(jd_skills) - set(resume_skills))
    score = round(len(matched) / len(jd_skills) * 100, 2) if jd_skills else 0

    print("\n📊 --- Resume vs JD Match Report ---")
    print(f"🧠 Match Score: {score}%\n")

    print("✅ Matched Skills:")
    for skill in matched:
        print(f" - {skill}")

    print("\n❌ Missing Skills:")
    for skill in missing:
        print(f" - {skill}")

    return {
        "score": score,
        "matched": matched,
        "missing": missing,
        "resume_skills": resume_skills,
        "jd_skills": jd_skills
    }


In [ ]:
# Pick first file from uploads
resume_path = list(resume_file.keys())[0]
jd_path = list(jd_file.keys())[0]

result = compare_resume_to_jd(resume_path, jd_path)
